# House Price Prediction — Linear Regression

Predicts house sale prices from square footage, number of bedrooms, and number of bathrooms.

Dataset: [Kaggle House Prices - Advanced Regression Techniques](https://www.kaggle.com/c/house-prices-advanced-regression-techniques/data)

Built to run as a Kaggle Notebook with the competition dataset attached.

## 1. Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

sns.set_style('whitegrid')

## 2. Load the data

In [ ]:
train = pd.read_csv('/kaggle/input/competitions/house-prices-advanced-regression-techniques/train.csv')
test = pd.read_csv('/kaggle/input/competitions/house-prices-advanced-regression-techniques/test.csv')

print(train.shape, test.shape)
train.head()

## 3. Select features

- `GrLivArea` — square footage (above-ground living area)
- `BedroomAbvGr` — number of bedrooms
- `TotalBath` — full baths + 0.5 x half baths

In [ ]:
# combine full and half baths into one "bathroom count"
train['TotalBath'] = train['FullBath'] + 0.5 * train['HalfBath']
test['TotalBath'] = test['FullBath'] + 0.5 * test['HalfBath']

features = ['GrLivArea', 'BedroomAbvGr', 'TotalBath']
train[features + ['SalePrice']].describe()

In [ ]:
train[features].isnull().sum()

## 4. Train / validation split

In [ ]:
X = train[features]
y = train['SalePrice']

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)
print(X_train.shape, X_val.shape)

## 5. Train the model

In [ ]:
model = LinearRegression()
model.fit(X_train, y_train)

print('Intercept:', model.intercept_)
for feature, coef in zip(features, model.coef_):
    print(f'{feature}: {coef:.2f}')

## 6. Evaluate

In [ ]:
y_pred = model.predict(X_val)

rmse = np.sqrt(mean_squared_error(y_val, y_pred))
mae = mean_absolute_error(y_val, y_pred)
r2 = r2_score(y_val, y_pred)

print(f'RMSE: {rmse:,.0f}')
print(f'MAE: {mae:,.0f}')
print(f'R2 score: {r2:.3f}')

## 7. Visualize actual vs. predicted

In [ ]:
plt.figure(figsize=(6, 6))
plt.scatter(y_val, y_pred, alpha=0.5)
plt.plot([y_val.min(), y_val.max()], [y_val.min(), y_val.max()], 'r--')
plt.xlabel('Actual sale price')
plt.ylabel('Predicted sale price')
plt.title('Actual vs predicted house prices')
plt.show()

## 8. Generate Kaggle submission

In [ ]:
test_features = test[features].fillna(X.median())
test['SalePrice'] = model.predict(test_features)

submission = test[['Id', 'SalePrice']]
submission.to_csv('submission.csv', index=False)